# 卷積神經網路

一張手寫數字是 28×28 個灰階像素，答案是 0 到 9 其中一個。上一份的網路每一層都是全連接：每個單元都接到前一層的每一個數。這一份先把圖攤平交給全連接網路，再換成卷積網路：用 3×3 的小核在圖上滑過去找筆畫，同一個核在每個位置都一樣。兩個網路用同一份資料訓練，再比參數數量、測試正確率，以及數字被往旁邊推之後還認不認得。

程式已經寫好。請 agent 改的格子，標題會寫「實驗」。下面這種格子只有你能寫，agent 不要填、不要改寫：


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>

</div>


## 給 agent

- 可以改標題含「實驗」的程式格，也可以依學生當下的問題加新的程式格。
- 不要改標題是「你的觀察」的格子，也不要替學生寫結論。
- 不要改兩個網路的層數、單元數、核數、學習率、epoch，除非學生明確說要比較這一次訓練的結果。
- 學生若只說「幫我做完」，先問他想先改卷積核，還是先改位移量，再改程式。


## 資料是 MNIST 的一小份

`data/mnist_small.npz` 從 MNIST 取出來：訓練 6000 張、測試 1000 張，每個數字一樣多，測試張都來自 MNIST 原本的測試集。MNIST 由 Yann LeCun、Corinna Cortes、Christopher J. C. Burges 整理，以 CC BY-SA 3.0 授權。

像素原本是 0 到 255，這裡除以 255 變成 0 到 1。下一格畫出前 10 張訓練圖，標題是答案。


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "Noto Sans TC", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Flatten, MaxPooling2D

tf.get_logger().setLevel("ERROR")

data = np.load(Path(__vsc_ipynb_file__).resolve().parent / "data" / "mnist_small.npz")
X_train = data["x_train"].astype("float32") / 255
y_train = data["y_train"]
X_test = data["x_test"].astype("float32") / 255
y_test = data["y_test"]
print("訓練", X_train.shape, "測試", X_test.shape)
print("像素最小/最大", X_train.min(), X_train.max())
print("每個數字的訓練張數", np.bincount(y_train))

fig, axes = plt.subplots(1, 10, figsize=(12, 1.8))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i], cmap="gray_r")
    ax.set_title(str(y_train[i]))
    ax.axis("off")
plt.show()


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">同一個數字，不同人寫起來差在哪：</li>
<li style="color:#1a1a1a;">字是不是都放在圖的中間：</li>
</ol>

</div>


## 一個卷積核在做什麼

卷積核是一小格數字，這裡是 3×3。把它蓋在圖的左上角，九個位置的像素各乘上核裡對應的數再加起來，得到一個數。往右移一格再算一次，一路滑完整張圖，這些數排起來就是特徵圖。核不能超出圖的邊，所以 28×28 的圖用 3×3 的核滑完，特徵圖是 26×26。

像素陣列裡，有墨的地方接近 1，空白是 0；圖上把 1 畫成黑色。下面這個核左欄是 1、右欄是 -1。特徵圖上算出來是正的畫成紅色，負的畫成藍色，0 是白色。


In [ ]:
# 實驗：改 kernel 的九個數，看特徵圖哪裡變紅、哪裡變藍
kernel = np.array(
    [
        [1, 0, -1],
        [1, 0, -1],
        [1, 0, -1],
    ],
    dtype="float32",
)

digit = X_train[0]
fmap = tf.nn.conv2d(
    digit[None, :, :, None],
    kernel[:, :, None, None],
    strides=1,
    padding="VALID",
)[0, :, :, 0].numpy()
print("輸入", digit.shape, "→ 特徵圖", fmap.shape)
print("特徵圖最小/最大", round(float(fmap.min()), 2), round(float(fmap.max()), 2))

limit = max(abs(float(fmap.min())), abs(float(fmap.max())), 1e-6)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
axes[0].imshow(digit, cmap="gray_r")
axes[0].set_title(f"input {y_train[0]}")
axes[1].imshow(kernel, cmap="bwr", vmin=-1, vmax=1)
axes[1].set_title("kernel")
shade = axes[2].imshow(fmap, cmap="bwr", vmin=-limit, vmax=limit)
axes[2].set_title("feature map")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.colorbar(shade, ax=axes[2])
plt.show()


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">特徵圖上的紅色出現在筆畫的哪一邊：</li>
<li style="color:#1a1a1a;">把核改成上排 1、下排 -1，紅色和藍色換到哪裡：</li>
<li style="color:#1a1a1a;">九個數全改成 <code style="color:#1a1a1a;">1/9</code>，特徵圖變成什麼樣子：</li>
</ol>

</div>


## 全連接和卷積各訓練一次

全連接網路先用 `Flatten` 把 28×28 攤成 784 個數，接一層 64 個單元的 ReLU，最後 10 個單元用 softmax 輸出每個數字的機會。

卷積網路先用 8 個 3×3 的核掃過整張圖，`MaxPooling2D` 把每 2×2 留下最大的那個、圖縮成一半；再用 16 個核掃一次、再縮一半；最後攤平接到同樣的 10 個 softmax 單元。核裡的數不是人給的，是訓練學出來的。

兩個網路都用 Adam、學習率 `0.001`、批次 64、跑 8 個 epoch。這些數字寫死，不在這一格改。每次重跑，正確率會差一點點。


In [ ]:
def build_dense():
    return Sequential(
        [
            tf.keras.Input(shape=(28, 28, 1)),
            Flatten(),
            Dense(64, activation="relu"),
            Dense(10, activation="softmax"),
        ]
    )

def build_cnn():
    return Sequential(
        [
            tf.keras.Input(shape=(28, 28, 1)),
            Conv2D(8, 3, activation="relu", name="conv1"),
            MaxPooling2D(),
            Conv2D(16, 3, activation="relu", name="conv2"),
            MaxPooling2D(),
            Flatten(),
            Dense(10, activation="softmax"),
        ]
    )

models = {}
for name, build in (("全連接", build_dense), ("卷積", build_cnn)):
    tf.random.set_seed(1234)
    model = build()
    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        metrics=["accuracy"],
    )
    model.fit(X_train[..., None], y_train, epochs=8, batch_size=64, verbose=0)
    _, acc = model.evaluate(X_test[..., None], y_test, verbose=0)
    models[name] = model
    print(f"{name}：參數 {model.count_params():,} 個，測試正確率 {acc:.3f}")

dense, cnn = models["全連接"], models["卷積"]


## 卷積網路學到的核

下一格畫出 `conv1` 學到的 8 個核，以及它們掃過一張測試圖得到的 8 張特徵圖。算法跟上面手寫的核一樣，只是多加上偏差、再經過 ReLU，所以負的地方都變成 0。

這 8 個核跟上一份的隱藏單元一樣沒有固定座位：重新訓練一次，第幾號核學到什麼可能會換。


In [ ]:
weights, bias = cnn.get_layer("conv1").get_weights()
print("conv1 權重形狀", weights.shape, "偏差形狀", bias.shape)

sample = X_test[0]
maps = tf.nn.relu(
    tf.nn.conv2d(sample[None, :, :, None], weights, strides=1, padding="VALID") + bias
)[0].numpy()

fig, axes = plt.subplots(2, 8, figsize=(14, 3.8))
for k in range(8):
    w = weights[:, :, 0, k]
    edge = max(abs(float(w.min())), abs(float(w.max())), 1e-6)
    axes[0, k].imshow(w, cmap="bwr", vmin=-edge, vmax=edge)
    axes[0, k].set_title(f"kernel {k}")
    axes[1, k].imshow(maps[:, :, k], cmap="gray_r")
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
axes[1, 0].set_ylabel(f"input {y_test[0]}")
plt.show()


## 把數字往旁邊推

兩個網路都只看過大致放在中間的字。下一格把每張測試圖整個往右推 `shift` 格，左邊空出來的補白，再測一次正確率。右圖畫出 `shift` 從 0 到 6，兩條線各掉多少。


In [ ]:
# 實驗：改 shift，看兩個網路的正確率各掉多少
shift = 3

def shift_right(images, pixels):
    moved = np.zeros_like(images)
    moved[:, :, pixels:] = images[:, :, : 28 - pixels]
    return moved

moved = shift_right(X_test, shift)
for name, model in models.items():
    _, acc = model.evaluate(moved[..., None], y_test, verbose=0)
    print(f"往右推 {shift} 格，{name}的測試正確率 {acc:.3f}")

steps = list(range(0, 7))
curves = {}
for name, model in models.items():
    curves[name] = [
        model.evaluate(shift_right(X_test, s)[..., None], y_test, verbose=0)[1] for s in steps
    ]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), gridspec_kw={"width_ratios": [1, 2]})
axes[0].imshow(moved[0], cmap="gray_r")
axes[0].set_title(f"shift {shift}")
axes[0].axis("off")
axes[1].plot(steps, curves["全連接"], marker="o", label="Dense")
axes[1].plot(steps, curves["卷積"], marker="s", label="CNN")
axes[1].axvline(shift, color="#999999", linewidth=1)
axes[1].set_xlabel("shift (pixels)")
axes[1].set_ylabel("test accuracy")
axes[1].set_ylim(0, 1)
axes[1].legend()
plt.show()


<div style="background:#fff6e0;border-left:6px solid #e09b00;padding:12px 16px;border-radius:4px;color:#1a1a1a;">

<h3 style="margin-top:0;color:#1a1a1a;">你的觀察</h3>
<ol style="color:#1a1a1a;list-style:decimal;padding-left:1.4em;">
<li style="color:#1a1a1a;">兩個網路的參數差幾倍，測試正確率差多少：</li>
<li style="color:#1a1a1a;">學到的 8 個核裡，有沒有跟你手寫的直邊核很像的：</li>
<li style="color:#1a1a1a;"><code style="color:#1a1a1a;">shift</code> 改大之後，哪一個網路的正確率掉得比較快：</li>
<li style="color:#1a1a1a;">同一個核在每個位置都一樣，這跟推過的字還認得出來有什麼關係：</li>
</ol>

</div>


## 這本沒有收進來的

不用 `tf.keras.datasets` 現場下載 MNIST，只用附在 `data/` 的 7000 張。不講 padding、stride、dropout，也不做資料擴增。學到的核每次訓練都不一樣，這本沒有存一組固定的權重。
